# Supervised selection of 50 items from the retrieval union

BM25+dense уже имеет test Recall@50 `0.8092`, но union их top-250 имеет
Recall около `0.94`. Значит, основная потеря сейчас происходит не при поиске
широкого пула, а при сжатии пула до 50 items. Этот notebook проверяет
supervised learning-to-rank как более дешёвую альтернативу ColBERT.

Защита от утечки:

- все точные query signatures validation полностью исключаются из click-history;
- исходный dev-fold дополнительно делится на fit/tune со стратификацией;
- набор признаков и число итераций выбираются только на tune;
- test открывается один раз для выбранного LTR-варианта;
- это четвёртый model-family look, поэтому primary threshold консервативно
  равен `0.05/4 = 0.0125`, а bootstrap CI имеет confidence 98.75%.

In [1]:
from __future__ import annotations

from collections import defaultdict
from dataclasses import asdict
from pathlib import Path
import gc
import json
import math
import re
import sys

import numpy as np
import pandas as pd
from catboost import CatBoostRanker, Pool
from sklearn.model_selection import StratifiedKFold, train_test_split

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from avito_retrieval.filters import requested_min_rating
from avito_retrieval.fusion import reciprocal_rank_fusion
from avito_retrieval.metrics import recall_at_k
from avito_retrieval.statistics import paired_recall_test

QUERY_COLUMNS = [
    "search_query", "search_location_id", "search_is_delivery_search",
    "search_infm_params_text", "search_category",
]
TOKEN_RE = re.compile(r"(?u)[\w]+(?:[-'][\w]+)*")
MISSING_RANK = 501
POOL_WIDTH = 200
PRIMARY_ALPHA = 0.0125
PRIMARY_CONFIDENCE = 0.9875


def clean(value: object) -> str:
    if value is None or pd.isna(value):
        return ""
    return " ".join(str(value).replace("ё", "е").lower().split())


def tokens(value: object) -> frozenset[str]:
    return frozenset(TOKEN_RE.findall(clean(value)))


def parse_rank(value: object) -> list[str]:
    return [] if value is None or pd.isna(value) or not str(value) else str(value).split()


def rank_map(values: list[str]) -> dict[str, int]:
    return {item_id: rank for rank, item_id in enumerate(values, 1)}


def subset(mapping: dict, ids: set[str]) -> dict:
    return {query_id: mapping[query_id] for query_id in ids}


def top_from_scores(frame: pd.DataFrame, score_column: str, k: int = 50) -> dict[str, list[str]]:
    ordered = frame.sort_values(
        ["query_key", score_column, "baseline_rank", "item_id"],
        ascending=[True, False, True, True],
    )
    return ordered.groupby("query_key", sort=False).head(k).groupby("query_key").item_id.agg(list).to_dict()


def overlap_features(query_tokens: frozenset[str], item_tokens: frozenset[str]) -> tuple[int, float, float]:
    overlap = len(query_tokens & item_tokens)
    coverage = overlap / max(len(query_tokens), 1)
    jaccard = overlap / max(len(query_tokens | item_tokens), 1)
    return overlap, coverage, jaccard


def safe_nonnegative(value: object) -> float:
    return max(float(value), 0.0) if value is not None and pd.notna(value) else 0.0

## 1. Данные, неизменный split и широкий candidate pool

Candidate pool — union top-200 BM25, уже выбранного dense global/local и
SPLADE q32/d192. SPLADE не принят как самостоятельный RRF-канал, но его
добавление в широкий pool стоит проверить: оно увеличивает oracle recall и не
заставляет ranker автоматически помещать sparse-кандидаты в top-50.
Ширина 200 выбрана как минимальный практически безопасный бюджет по dev-oracle
при локальном лимите памяти; test-oracle до фиксации модели не используется.

In [2]:
manifest = pd.read_parquet(ROOT / "artifacts/validation/manifest.parquet").copy()
labels = pd.read_parquet(ROOT / "artifacts/validation/labels.parquet")
benchmark = pd.read_parquet(ROOT / "dataset/benchmark_queries.parquet")
items = pd.read_parquet(ROOT / "dataset/benchmark_items.parquet")
train = pd.read_parquet(ROOT / "dataset/train.parquet")

bm25 = pd.read_parquet(ROOT / "artifacts/rankings/bm25_rankings.parquet")
dense = pd.read_parquet(ROOT / "artifacts/dense_kaggle/dense_rankings.parquet")
splade = pd.read_parquet(ROOT / "artifacts/splade_kaggle/splade_rankings.parquet")
rankings = bm25.merge(dense, on=["query_key", "split"], validate="one_to_one").merge(
    splade, on=["query_key", "split"], validate="one_to_one"
)

relevant = labels.groupby("eval_query_id").item_id.agg(set).to_dict()
manifest["fold"] = -1
outer = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
for fold, (_, indices) in enumerate(outer.split(manifest, manifest.primary_stratum)):
    manifest.loc[indices, "fold"] = fold
dev_ids = set(manifest.loc[manifest.fold.eq(0), "eval_query_id"])
test_ids = set(manifest.loc[manifest.fold.eq(1), "eval_query_id"])

dev_table = manifest[manifest.eval_query_id.isin(dev_ids)]
fit_values, tune_values = train_test_split(
    dev_table.eval_query_id,
    test_size=0.25,
    random_state=42,
    stratify=dev_table.primary_stratum,
)
fit_ids, tune_ids = set(fit_values), set(tune_values)
print({"fit": len(fit_ids), "tune": len(tune_ids), "test": len(test_ids)})


def channel_lists(row: pd.Series) -> dict[str, list[str]]:
    bm25_rank = parse_rank(row.bm25)
    dense_rank = reciprocal_rank_fusion(
        [parse_rank(row.dense_global), parse_rank(row.dense_local)],
        weights=[1.0, 1.15], top_k=250,
    )
    splade_rank = reciprocal_rank_fusion(
        [parse_rank(row.splade_q32_d192_global), parse_rank(row.splade_q32_d192_local)],
        weights=[1.0, 2.0], rrf_k=60, top_k=250,
    )
    baseline = reciprocal_rank_fusion(
        [bm25_rank, dense_rank], weights=[1.0, 1.25], rrf_k=20, top_k=250
    )
    return {"bm25": bm25_rank, "dense": dense_rank, "splade": splade_rank, "baseline": baseline}


validation_rankings = rankings[rankings.split.eq("validation")].set_index("query_key")
channel_cache = {query_id: channel_lists(row) for query_id, row in validation_rankings.iterrows()}
baseline_pred = {query_id: values["baseline"][:50] for query_id, values in channel_cache.items()}
union_bd = {
    query_id: list(dict.fromkeys([*values["bm25"][:POOL_WIDTH], *values["dense"][:POOL_WIDTH]]))
    for query_id, values in channel_cache.items()
}
union_all = {
    query_id: list(dict.fromkeys([*union_bd[query_id], *values["splade"][:POOL_WIDTH]]))
    for query_id, values in channel_cache.items()
}
print("dev oracle BM25+dense:", recall_at_k(subset(union_bd, dev_ids), subset(relevant, dev_ids), k=2 * POOL_WIDTH))
print("dev oracle +SPLADE:", recall_at_k(subset(union_all, dev_ids), subset(relevant, dev_ids), k=3 * POOL_WIDTH))

{'fit': 919, 'tune': 307, 'test': 1226}


dev oracle BM25+dense: 0.9352909189777053
dev oracle +SPLADE: 0.9434475258292551


## 2. Leakage-safe history и pair features

История вычисляется после anti-join по всем пяти query-полям: ни один клик
validation query signature не может стать признаком самого себя. Exact
`search_query → item/microcategory` history остаётся допустимым production
сигналом для повторяющихся формулировок. Три вложенных feature sets отделяют
вклад rank channels, content/metadata и истории.

In [3]:
marked = train.merge(
    manifest[QUERY_COLUMNS].drop_duplicates().assign(_validation_query=1),
    on=QUERY_COLUMNS, how="left",
)
history_train = marked[marked._validation_query.isna()].drop(columns="_validation_query").copy()
history_train["query_norm"] = history_train.search_query.map(clean)
item_pop = history_train.item_id.astype(str).value_counts().to_dict()
query_item_pop = history_train.groupby(["query_norm", "item_id"]).size().to_dict()
query_microcat_pop = history_train.groupby(["query_norm", "item_microcat_id"]).size().to_dict()
assert len(history_train) + marked._validation_query.notna().sum() == len(train)

item_frame = items.copy()
item_frame["item_id"] = item_frame.item_id.astype(str)
item_frame = item_frame.set_index("item_id", drop=False)
item_token_cache = {
    item_id: (
        tokens(row.item_title_raw), tokens(row.item_infm_params_text),
        tokens(row.item_description_raw),
    )
    for item_id, row in item_frame.iterrows()
}


def build_features(query_frame: pd.DataFrame, ranking_frame: pd.DataFrame) -> pd.DataFrame:
    query_by_key = query_frame.set_index("query_key")
    records: list[dict[str, object]] = []
    for position, (query_id, rank_row) in enumerate(ranking_frame.iterrows(), 1):
        query = query_by_key.loc[query_id]
        channels = channel_lists(rank_row)
        maps = {name: rank_map(values) for name, values in channels.items()}
        candidate_ids = list(dict.fromkeys([
            *channels["bm25"][:POOL_WIDTH], *channels["dense"][:POOL_WIDTH],
            *channels["splade"][:POOL_WIDTH],
        ]))
        q_words = tokens(f"{query.search_query} {query.search_infm_params_text}")
        filter_words = tokens(query.search_infm_params_text)
        q_norm = clean(query.search_query)
        min_rating = requested_min_rating(query.search_infm_params_text)
        for item_id in candidate_ids:
            item = item_frame.loc[item_id]
            title_words, param_words, description_words = item_token_cache[item_id]
            title_overlap = overlap_features(q_words, title_words)
            param_overlap = overlap_features(q_words, param_words)
            desc_overlap = overlap_features(q_words, description_words)
            filter_overlap = overlap_features(filter_words, param_words)
            channel_ranks = {name: maps[name].get(item_id, MISSING_RANK) for name in maps}
            source_count = sum(channel_ranks[name] <= POOL_WIDTH for name in ("bm25", "dense", "splade"))
            rating = float(item.item_rating) if pd.notna(item.item_rating) else -1.0
            records.append({
                "query_key": str(query_id), "item_id": item_id,
                "label": int(item_id in relevant.get(str(query_id), set())),
                "bm25_rank": channel_ranks["bm25"], "dense_rank": channel_ranks["dense"],
                "splade_rank": channel_ranks["splade"], "baseline_rank": channel_ranks["baseline"],
                "rr_bm25": 1.0 / (20 + channel_ranks["bm25"]),
                "rr_dense": 1.25 / (20 + channel_ranks["dense"]),
                "rr_splade": 0.5 / (20 + channel_ranks["splade"]),
                "min_source_rank": min(channel_ranks["bm25"], channel_ranks["dense"], channel_ranks["splade"]),
                "source_count": source_count,
                "title_overlap": title_overlap[0], "title_coverage": title_overlap[1], "title_jaccard": title_overlap[2],
                "param_overlap": param_overlap[0], "param_coverage": param_overlap[1], "param_jaccard": param_overlap[2],
                "desc_overlap": desc_overlap[0], "desc_coverage": desc_overlap[1], "desc_jaccard": desc_overlap[2],
                "filter_param_overlap": filter_overlap[0], "filter_param_coverage": filter_overlap[1],
                "exact_query_in_title": int(bool(q_norm) and q_norm in clean(item.item_title_raw)),
                "exact_query_in_description": int(bool(q_norm) and q_norm in clean(item.item_description_raw)),
                "location_match": int(int(query.search_location_id) == int(item.item_location_id)),
                "category_match": int(int(query.search_category) in (0, int(item.item_category_id))),
                "rating": rating,
                "rating_margin": rating - min_rating if min_rating is not None else 0.0,
                "log_reviews": math.log1p(safe_nonnegative(item.item_rating_reviews_count)),
                "log_price": math.log1p(safe_nonnegative(item.item_price)),
                "phone_hidden": int(item.item_is_phone_hidden == 1),
                "message_forbidden": int(item.item_is_message_forbidden == 1),
                "title_tokens": len(title_words), "description_tokens": len(description_words),
                "log_item_clicks": math.log1p(item_pop.get(item_id, 0)),
                "log_query_item_clicks": math.log1p(query_item_pop.get((q_norm, item_id), 0)),
                "log_query_microcat_clicks": math.log1p(query_microcat_pop.get((q_norm, item.item_microcat_id), 0)),
            })
        if position % 250 == 0:
            print(f"features {position}/{len(ranking_frame)}")
    return pd.DataFrame.from_records(records)


validation_queries = manifest.rename(columns={"eval_query_id": "query_key"})
validation_features = build_features(validation_queries, validation_rankings)
for column in validation_features.select_dtypes(include=["float64"]).columns:
    validation_features[column] = validation_features[column].astype("float32")
for column in validation_features.select_dtypes(include=["int64"]).columns:
    validation_features[column] = pd.to_numeric(validation_features[column], downcast="integer")
print(validation_features.shape, "positives in pool=", int(validation_features.label.sum()))
# The token-set cache dominates RAM and is not needed while CatBoost trains.
del item_token_cache
gc.collect()

features 250/2452


features 500/2452


features 750/2452


features 1000/2452


features 1250/2452


...

features 1750/2452


features 2000/2452


features 2250/2452


(1069675, 38) positives in pool= 2537


0

## 3. Nested dev selection

CatBoostRanker оптимизирует pairwise порядок внутри query group. Для обучения
оставляем все positives и максимум 200 самых трудных negatives по минимальному
рангу каналов: случайные далёкие negatives почти не учат границу top-50, но
резко увеличивают время. `rank`, `content`, `history` — вложенные ablations.
Каждая модель использует одинаковые fit/tune groups и early stopping.

In [4]:
rank_features = [
    "bm25_rank", "dense_rank", "splade_rank", "baseline_rank",
    "rr_bm25", "rr_dense", "rr_splade", "min_source_rank", "source_count",
]
content_features = rank_features + [
    "title_overlap", "title_coverage", "title_jaccard",
    "param_overlap", "param_coverage", "param_jaccard",
    "desc_overlap", "desc_coverage", "desc_jaccard",
    "filter_param_overlap", "filter_param_coverage",
    "exact_query_in_title", "exact_query_in_description",
    "location_match", "category_match", "rating", "rating_margin",
    "log_reviews", "log_price", "phone_hidden", "message_forbidden",
    "title_tokens", "description_tokens",
]
history_features = content_features + [
    "log_item_clicks", "log_query_item_clicks", "log_query_microcat_clicks",
]
feature_sets = {"rank": rank_features, "content": content_features, "history": history_features}


def hard_negative_subset(frame: pd.DataFrame, ids: set[str], max_rows: int = 201) -> pd.DataFrame:
    selected = frame[frame.query_key.isin(ids)].sort_values(
        ["query_key", "label", "min_source_rank"], ascending=[True, False, True]
    )
    return selected.groupby("query_key", sort=False).head(max_rows).reset_index(drop=True)


def make_pool(frame: pd.DataFrame, features: list[str]) -> Pool:
    group_id = pd.factorize(frame.query_key, sort=False)[0].astype("int32")
    return Pool(frame[features], label=frame.label, group_id=group_id)


fit_frame = hard_negative_subset(validation_features, fit_ids)
tune_frame = validation_features[validation_features.query_key.isin(tune_ids)].copy()
tune_predictions = {}
selection_rows = []
for name, features in feature_sets.items():
    model = CatBoostRanker(
        loss_function="YetiRankPairwise", eval_metric="NDCG:top=50",
        iterations=500, depth=7, learning_rate=0.05,
        random_seed=42, thread_count=-1, verbose=100,
        od_type="Iter", od_wait=60, allow_writing_files=False,
    )
    model.fit(make_pool(fit_frame, features), eval_set=make_pool(tune_frame, features))
    scored = tune_frame[["query_key", "item_id", "baseline_rank"]].copy()
    scored["score"] = model.predict(tune_frame[features])
    prediction = top_from_scores(scored, "score")
    tune_predictions[name] = prediction
    selection_rows.append({
        "features": name,
        "best_iteration": max(model.get_best_iteration() + 1, 1),
        "tune_recall": recall_at_k(prediction, subset(relevant, tune_ids)),
    })
    del model, scored
    gc.collect()
selection = pd.DataFrame(selection_rows).sort_values(["tune_recall", "features"], ascending=[False, True])
display(selection)
selected_features_name = str(selection.iloc[0].features)
selected_features = feature_sets[selected_features_name]
selected_iterations = int(selection.iloc[0].best_iteration)
print("Selected on tune:", selected_features_name, selected_iterations)

tune_ablation = {
    "content_vs_rank": asdict(paired_recall_test(
        tune_predictions["content"], tune_predictions["rank"], relevant,
        query_ids=sorted(tune_ids), n_resamples=20_000,
    )),
    "history_vs_content": asdict(paired_recall_test(
        tune_predictions["history"], tune_predictions["content"], relevant,
        query_ids=sorted(tune_ids), n_resamples=20_000,
    )),
}
display(pd.DataFrame(tune_ablation).T)

0:	test: 0.3628465	best: 0.3628465 (0)	total: 192ms	remaining: 1m 35s


100:	test: 0.4844856	best: 0.4883852 (57)	total: 14.5s	remaining: 57.3s


Stopped by overfitting detector  (60 iterations wait)

bestTest = 0.4883851782
bestIteration = 57

Shrink model to first 58 iterations.


0:	test: 0.4001732	best: 0.4001732 (0)	total: 173ms	remaining: 1m 26s


100:	test: 0.5264956	best: 0.5311110 (91)	total: 16.4s	remaining: 1m 4s


...

200:	test: 0.5992289	best: 0.6023994 (194)	total: 1m 5s	remaining: 1m 38s


Stopped by overfitting detector  (60 iterations wait)

bestTest = 0.602399428
bestIteration = 194

Shrink model to first 195 iterations.


,features,best_iteration,tune_recall
2,history,195,0.824104
1,content,92,0.795331
0,rank,58,0.743757


Selected on tune: history 195


,mean_delta,ci_low,ci_high,p_value_greater,n_queries
content_vs_rank,0.051574,0.029316,0.076547,0.000050,307.0
history_vs_content,0.028773,0.001086,0.058089,0.030598,307.0


## 4. Единственная primary-оценка на test

После выбора feature set и iteration budget модель переобучается на всём dev.
На test сравнивается только с зафиксированным BM25+dense RRF. Принятие требует
одновременно положительный эффект, 98.75% CI выше нуля, `p<0.0125` и итоговый
test Recall@50 выше baseline. Отдельно показывается расстояние до цели 0.9.

In [5]:
dev_train = hard_negative_subset(validation_features, dev_ids)
final_model = CatBoostRanker(
    loss_function="YetiRankPairwise", iterations=selected_iterations,
    depth=7, learning_rate=0.05, random_seed=42, thread_count=-1,
    verbose=100, allow_writing_files=False,
)
final_model.fit(make_pool(dev_train, selected_features))
test_frame = validation_features[validation_features.query_key.isin(test_ids)].copy()
test_frame["ltr_score"] = final_model.predict(test_frame[selected_features])
ltr_test = top_from_scores(test_frame, "ltr_score")

primary = asdict(paired_recall_test(
    ltr_test, baseline_pred, relevant, query_ids=sorted(test_ids),
    confidence=PRIMARY_CONFIDENCE,
))
baseline_test = recall_at_k(subset(baseline_pred, test_ids), subset(relevant, test_ids))
ltr_test_recall = recall_at_k(ltr_test, subset(relevant, test_ids))
primary_confirmed = bool(
    primary["mean_delta"] > 0 and primary["ci_low"] > 0
    and primary["p_value_greater"] < PRIMARY_ALPHA
    and ltr_test_recall > baseline_test
)
display(pd.DataFrame([
    {"method": "BM25+dense", "test_recall": baseline_test},
    {"method": f"LTR-{selected_features_name}", "test_recall": ltr_test_recall},
]))
display(pd.DataFrame({"LTR_vs_baseline": primary}).T)
print("distance to 0.9:", 0.9 - ltr_test_recall, "accepted:", primary_confirmed)

0:	total: 775ms	remaining: 2m 30s


100:	total: 43.7s	remaining: 40.6s


194:	total: 1m 19s	remaining: 0us


,method,test_recall
0,BM25+dense,0.809203
1,LTR-history,0.840579


,mean_delta,ci_low,ci_high,p_value_greater,n_queries
LTR_vs_baseline,0.031376,0.010536,0.052787,0.00025,1226.0


distance to 0.9: 0.05942088091354003 accepted: True


## 5. Pool/segment diagnostics и benchmark answer

Если LTR принят, production model обучается на всех validation queries тем же
iteration budget. Benchmark признаки строятся идентично, без labels. Файл
`answer_ltr.csv` сохраняется для аудита всегда; основной `answer.csv`
заменяется только при выполнении primary-критерия.

In [6]:
pool_test = {
    "pool_width_per_channel": POOL_WIDTH,
    "BM25+dense_union": recall_at_k(subset(union_bd, test_ids), subset(relevant, test_ids), k=2 * POOL_WIDTH),
    "BM25+dense+SPLADE_union": recall_at_k(subset(union_all, test_ids), subset(relevant, test_ids), k=3 * POOL_WIDTH),
}
slice_rows = []
for column in ["has_filters", "local_pool_bucket", "query_frequency_bucket", "positive_bucket"]:
    for value, group in manifest[manifest.eval_query_id.isin(test_ids)].groupby(column):
        ids = set(group.eval_query_id)
        base = recall_at_k(subset(baseline_pred, ids), subset(relevant, ids))
        new = recall_at_k(subset(ltr_test, ids), subset(relevant, ids))
        slice_rows.append({"slice": column, "value": str(value), "queries": len(ids),
                           "baseline": base, "ltr": new, "delta": new - base})
slices = pd.DataFrame(slice_rows)
display(slices)

all_train = hard_negative_subset(validation_features, set(relevant))
production_model = CatBoostRanker(
    loss_function="YetiRankPairwise", iterations=selected_iterations,
    depth=7, learning_rate=0.05, random_seed=42, thread_count=-1,
    verbose=100, allow_writing_files=False,
)
production_model.fit(make_pool(all_train, selected_features))

benchmark_rankings = rankings[rankings.split.eq("benchmark")].set_index("query_key")
# Validation pair table is the dominant RAM consumer. It is no longer needed
# after the production model is fitted; keep only compact predictions/metrics.
del validation_features, fit_frame, tune_frame, dev_train, test_frame, all_train
del train, marked, history_train, final_model
gc.collect()
item_token_cache = {
    item_id: (
        tokens(row.item_title_raw), tokens(row.item_infm_params_text),
        tokens(row.item_description_raw),
    )
    for item_id, row in item_frame.iterrows()
}
benchmark_queries = benchmark.copy()
benchmark_queries["query_key"] = benchmark_queries.query_id.astype(str)
benchmark_features = build_features(benchmark_queries, benchmark_rankings)
for column in benchmark_features.select_dtypes(include=["float64"]).columns:
    benchmark_features[column] = benchmark_features[column].astype("float32")
for column in benchmark_features.select_dtypes(include=["int64"]).columns:
    benchmark_features[column] = pd.to_numeric(benchmark_features[column], downcast="integer")
benchmark_features["ltr_score"] = production_model.predict(benchmark_features[selected_features])
benchmark_pred = top_from_scores(benchmark_features, "ltr_score")
answer = pd.DataFrame({
    "query_id": benchmark.query_id.astype(str),
    "answer": [" ".join(benchmark_pred[str(query_id)]) for query_id in benchmark.query_id],
})
valid_items = set(items.item_id.astype(str))
assert len(answer) == len(benchmark) and answer.query_id.is_unique
for value in answer.answer:
    ids = value.split()
    assert len(ids) == 50 and len(ids) == len(set(ids)) and set(ids) <= valid_items
    assert all(re.fullmatch(r"[0-9a-f]{16}", item_id) for item_id in ids)
answer.to_csv(ROOT / "answer_ltr.csv", index=False, encoding="utf-8")
if primary_confirmed:
    answer.to_csv(ROOT / "answer.csv", index=False, encoding="utf-8")

metrics = {
    "candidate_pool": pool_test,
    "selection": selection.to_dict("records"),
    "selected_features": selected_features_name,
    "selected_iterations": selected_iterations,
    "tune_ablations": tune_ablation,
    "baseline_test_recall": baseline_test,
    "ltr_test_recall": ltr_test_recall,
    "distance_to_0_9": 0.9 - ltr_test_recall,
    "primary_alpha": PRIMARY_ALPHA,
    "bootstrap_confidence": PRIMARY_CONFIDENCE,
    "primary_test": primary,
    "primary_confirmed": primary_confirmed,
    "slice_metrics": slices.to_dict("records"),
}
(ROOT / "reports/ltr_metrics.json").write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8"
)
display(answer.head())
print("answer_ltr.csv validated; promoted to answer.csv:", primary_confirmed)

,slice,value,queries,baseline,ltr,delta
0,has_filters,False,773,0.821151,0.854420,0.033269
1,has_filters,True,453,0.788815,0.816961,0.028146
2,local_pool_bucket,0,213,0.462441,0.556338,0.093897
3,local_pool_bucket,1-99,21,0.857143,0.761905,-0.095238
4,local_pool_bucket,100-499,106,0.867925,0.877358,0.009434
5,local_pool_bucket,10000+,272,0.794118,0.833824,0.039706
6,local_pool_bucket,2000-9999,277,0.908243,0.948556,0.040313
7,local_pool_bucket,500-1999,337,0.937685,0.930267,-0.007418
8,query_frequency_bucket,1,189,0.867725,0.851852,-0.015873
9,query_frequency_bucket,2-5,231,0.787879,0.800866,0.012987


0:	total: 271ms	remaining: 52.6s


100:	total: 52.1s	remaining: 48.5s


194:	total: 1m 21s	remaining: 0us


features 250/2452


...

features 1750/2452


features 2000/2452


features 2250/2452


,query_id,answer
0,70DfDUpwjxB4lzFd,deb58d7ad8c27d06 14d535285758af89 6fc4db453f87...
1,JTrdTaZJvSiLPkXj,1c6d2079e07f4497 8703b2a0442c5d25 cbeccbecb1fb...
2,LZCZNoVG4AFUkVRJ,d8fce513e4f000a7 dab52187b4500d9b 3f89b8062dc8...
3,660ac9QVtXkRxZC3,67b2cd847669eb36 af91ec4a29b66636 bd2cb4500fad...
4,YgHcM9MVbxKnxD1e,1a62634394544781 13f58fe9542bdb1a 615c73ea4c3b...


answer_ltr.csv validated; promoted to answer.csv: True
